# 04 — Robot Parameter Analysis

Önce baseline sonuçlarını doğrular, ardından yalnızca geliştirme döneminde kontrollü bir sinyal parametresi taraması yaparız. Holdout dönemi parametre seçiminde kullanılmaz.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.config import StrategyConfig, PortfolioConfig
from src.features import add_indicators
from src.signals import build_market_regime, add_robot_scores
from src.backtest import run_portfolio_backtest
from src.metrics import (
    portfolio_metrics,
    yearly_performance,
)
from src.experiments import (
    evaluate_strategy,
    run_strategy_grid,
    apply_robustness_filters,
    compare_periods,
)

sns.set_theme(style="whitegrid")


## 1. Verileri ve özellikleri hazırla


In [ ]:
stock_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bist100_robot_clean.parquet"
)

market_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xu100_robot_clean.parquet"
)

stock_features = add_indicators(stock_prices)
market_features = add_indicators(market_prices)
market_regime = build_market_regime(market_features)

print("Hisse özelliği:", stock_features.shape)
print("Hisse sayısı:", stock_features["Ticker"].nunique())
print(
    "Tarih aralığı:",
    stock_features["Date"].min(),
    "→",
    stock_features["Date"].max(),
)


## 2. Baseline sonucu yeniden üret ve doğrula


In [ ]:
base_strategy = StrategyConfig()
base_portfolio = PortfolioConfig()

baseline_scored = add_robot_scores(
    stock_features,
    market_regime,
    base_strategy,
    include_reasons=False,
)

baseline_equity, baseline_trades = run_portfolio_backtest(
    baseline_scored,
    base_strategy,
    base_portfolio,
)

baseline_metrics = portfolio_metrics(
    baseline_equity,
    baseline_trades,
)

display(pd.DataFrame([baseline_metrics]))
display(yearly_performance(baseline_equity))


In [ ]:
assert not baseline_equity.empty
assert baseline_equity["Equity"].gt(0).all()
assert baseline_equity["Open_Positions"].le(
    base_portfolio.max_positions
).all()

if not baseline_trades.empty:
    assert baseline_trades["Shares"].gt(0).all()
    assert (
        baseline_trades["Exit_Date"]
        >= baseline_trades["Entry_Date"]
    ).all()

print(
    "Outlier işlem:",
    int(
        baseline_trades.get(
            "Is_Outlier",
            pd.Series(dtype=bool),
        ).sum()
    ),
)

display(
    baseline_trades.nsmallest(
        10,
        "Return_%",
    )
)

display(
    baseline_trades.nlargest(
        10,
        "Return_%",
    )
)


## 3. Zaman ayrımı

- Geliştirme: 2018–2022
- Doğrulama: 2023–2024
- Holdout: 2025–son veri

Holdout dönemi yalnızca strateji seçildikten sonra bir kez çalıştırılmalıdır.


In [ ]:
PERIODS = {
    "Development": ("2018-01-01", "2022-12-31"),
    "Validation": ("2023-01-01", "2024-12-31"),
    "Holdout": (
        "2025-01-01",
        stock_features["Date"].max().strftime("%Y-%m-%d"),
    ),
}

baseline_period_records = []

for period_name, (start, end) in PERIODS.items():
    metrics, _, _ = evaluate_strategy(
        stock_features=stock_features,
        market_regime=market_regime,
        strategy_config=base_strategy,
        portfolio_config=base_portfolio,
        start=start,
        end=end,
    )

    metrics["Period"] = period_name
    baseline_period_records.append(metrics)

baseline_period_results = pd.DataFrame(
    baseline_period_records
)

display(
    baseline_period_results[
        [
            "Period",
            "CAGR_%",
            "Max_Drawdown_%",
            "Profit_Factor",
            "Win_Rate_%",
            "Sharpe",
            "Calmar",
            "Trade_Count",
        ]
    ]
)


## 4. Aşama 1 — Sinyal filtresi taraması

İlk aşamada stop ve portföy riskini sabit tutuyoruz. Yalnızca skor, ADX ve hacim eşiğini test ediyoruz.


In [ ]:
signal_grid = {
    "buy_score": [10, 11, 12],
    "minimum_adx": [18.0, 20.0, 22.0],
    "volume_multiplier": [1.10, 1.30, 1.50],
}

development_results = run_strategy_grid(
    stock_features=stock_features,
    market_regime=market_regime,
    base_strategy=base_strategy,
    portfolio_config=base_portfolio,
    parameter_grid=signal_grid,
    start=PERIODS["Development"][0],
    end=PERIODS["Development"][1],
)

development_results.to_csv(
    PROJECT_ROOT
    / "results"
    / "signal_grid_development.csv",
    index=False,
)

print("Deney sayısı:", len(development_results))
display(
    development_results[
        development_results["Status"].eq("OK")
    ].sort_values(
        ["Calmar", "CAGR_%"],
        ascending=False,
    ).head(15)
)


In [ ]:
robust_development = apply_robustness_filters(
    development_results,
    minimum_trades=40,
    maximum_drawdown_limit=-35.0,
    minimum_profit_factor=1.10,
)

selected_development = robust_development.head(5).copy()

display(
    selected_development[
        [
            "Experiment_ID",
            "Strategy_buy_score",
            "Strategy_minimum_adx",
            "Strategy_volume_multiplier",
            "CAGR_%",
            "Max_Drawdown_%",
            "Profit_Factor",
            "Sharpe",
            "Calmar",
            "Trade_Count",
        ]
    ]
)


## 5. İlk beş konfigürasyonu doğrulama döneminde karşılaştır


In [ ]:
parameter_columns = [
    "Strategy_buy_score",
    "Strategy_minimum_adx",
    "Strategy_volume_multiplier",
]

selected_comparison = compare_periods(
    stock_features=stock_features,
    market_regime=market_regime,
    configurations=selected_development,
    base_strategy=base_strategy,
    portfolio_config=base_portfolio,
    periods={
        "Development": PERIODS["Development"],
        "Validation": PERIODS["Validation"],
    },
    parameter_columns=parameter_columns,
)

comparison_view = selected_comparison[
    [
        "Selected_Config",
        "Period_Name",
        "Strategy_buy_score",
        "Strategy_minimum_adx",
        "Strategy_volume_multiplier",
        "CAGR_%",
        "Max_Drawdown_%",
        "Profit_Factor",
        "Sharpe",
        "Calmar",
        "Trade_Count",
    ]
].sort_values(
    ["Selected_Config", "Period_Name"]
)

display(comparison_view)

selected_comparison.to_csv(
    PROJECT_ROOT
    / "results"
    / "signal_grid_selected_validation.csv",
    index=False,
)


Bu aşamada holdout sonucu üzerinden seçim yapma. Geliştirme ve doğrulama dönemlerinin ikisinde de dengeli kalan konfigürasyonu seçtikten sonra çıkış parametreleri test edilecektir.


In [ ]:
baseline_period_results[
    [
        "Period",
        "CAGR_%",
        "Max_Drawdown_%",
        "Profit_Factor",
        "Win_Rate_%",
        "Sharpe",
        "Calmar",
        "Trade_Count",
    ]
]

In [ ]:
comparison_view